# AlphaTrend Backtest — 5H Bars, Long-Only, Compounding Portfolio

**Tickers:** SNDK, LITE, WDC, MU, STX  
**Timeframe:** 5H (built by grouping 5 consecutive 1H bars)  
**History:** ~9 months  
**Indicator:** AlphaTrend (Pine v5 port — `coeff=1`, `AP=14`, **RSI mode**, i.e. `novolumedata=true`)  
**Entry/Exit:** BUY signal → next-bar open. SELL signal → next-bar open. Long-only.  

**Portfolio model:**  
- Single shared cash balance, starts at **$10,000**.  
- Only **one position** open at a time across all tickers.  
- When flat, the **earliest** BUY signal across all tickers is taken.  
- Position size = `current_balance × 1.8` (notional). PnL on the notional updates the cash balance — **compounding**.  
- Other tickers' BUY signals while in position are ignored.  
- Exit only on the **same ticker**'s SELL signal.  

**Commission:** 0.15% per side, on traded notional.

> **Google Colab:** *File → Upload notebook* → pick this `.ipynb` → **Runtime → Run all**. The next code cell installs deps and sets the plotly renderer for Colab.

In [ ]:
# --- Colab / local setup ---
# Installs are quiet and idempotent. yfinance is not pre-installed on Colab.
import sys, subprocess
for pkg in ('yfinance', 'plotly'):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        check=False,
    )

import plotly.io as pio
try:
    import google.colab  # noqa: F401
    pio.renderers.default = 'colab'
    IN_COLAB = True
except ImportError:
    pio.renderers.default = 'notebook_connected'
    IN_COLAB = False

print('Environment:', 'Google Colab' if IN_COLAB else 'local Jupyter')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

TICKERS     = ['SNDK', 'LITE', 'WDC', 'MU', 'STX']
HISTORY_D   = 270           # ~9 months
BARS_PER_5H = 5             # group 5 hourly bars into one 5H bar
COEFF       = 1.0           # AlphaTrend Multiplier
AP          = 14            # AlphaTrend Common Period
START_BAL   = 10_000.0      # starting cash
LEVERAGE    = 1.8           # notional = balance × leverage on each entry
COMM_RATE   = 0.0015        # 0.15% per side, applied to traded notional
USE_MFI     = False         # RSI mode (Pine: novolumedata=true). Set True for MFI mode.

In [ ]:
def fetch_1h(ticker, days=HISTORY_D):
    end = pd.Timestamp.utcnow().tz_localize(None)
    start = end - pd.Timedelta(days=days)
    df = yf.download(
        ticker, start=start, end=end,
        interval='1h', auto_adjust=True, progress=False, threads=False,
    )
    if df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    return df

def resample_to_5h(df_1h, n=BARS_PER_5H):
    if df_1h.empty:
        return df_1h
    grp = np.arange(len(df_1h)) // n
    out = pd.DataFrame({
        'Open':   df_1h['Open'].groupby(grp).first(),
        'High':   df_1h['High'].groupby(grp).max(),
        'Low':    df_1h['Low'].groupby(grp).min(),
        'Close':  df_1h['Close'].groupby(grp).last(),
        'Volume': df_1h['Volume'].groupby(grp).sum(),
    })
    out.index = df_1h.index.to_series().groupby(grp).first().values
    return out

In [ ]:
def true_range(df):
    # Pine `ta.tr` returns NA on the first bar (default handle_na=false).
    prev_close = df['Close'].shift(1)
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - prev_close).abs(),
        (df['Low']  - prev_close).abs(),
    ], axis=1).max(axis=1)
    tr.iloc[0] = np.nan
    return tr

def mfi(df, period=AP):
    # Pine `ta.mfi(hlc3, AP)`
    tp = (df['High'] + df['Low'] + df['Close']) / 3.0
    raw_mf = tp * df['Volume']
    delta = tp.diff()
    pos_mf = raw_mf.where(delta > 0, 0.0)
    neg_mf = raw_mf.where(delta < 0, 0.0)
    pos_sum = pos_mf.rolling(period).sum()
    neg_sum = neg_mf.rolling(period).sum()
    mfr = pos_sum / neg_sum.replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + mfr))

def rsi(series, period=AP):
    # Pine `ta.rsi(src, AP)` — RMA (Wilder) smoothing
    delta = series.diff()
    up = delta.clip(lower=0)
    dn = (-delta).clip(lower=0)
    avg_up = up.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_dn = dn.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_up / avg_dn.replace(0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))

def alphatrend(df, coeff=COEFF, ap=AP, use_mfi=USE_MFI):
    # Direct port of:
    #   ATR        = ta.sma(ta.tr, AP)
    #   upT        = low  - ATR*coeff
    #   downT      = high + ATR*coeff
    #   AlphaTrend = (mfi>=50)
    #                  ? (upT  < nz(AT[1]) ? nz(AT[1]) : upT)
    #                  : (downT> nz(AT[1]) ? nz(AT[1]) : downT)
    tr  = true_range(df)
    atr = tr.rolling(ap).mean()
    up_t   = df['Low']  - atr * coeff
    down_t = df['High'] + atr * coeff
    osc = mfi(df, ap) if use_mfi else rsi(df['Close'], ap)

    out  = np.full(len(df), np.nan)
    prev = 0.0  # nz(AlphaTrend[1]) defaults to 0
    for i in range(len(df)):
        ut = up_t.iloc[i]
        dt = down_t.iloc[i]
        if np.isnan(ut) or np.isnan(dt):
            continue
        o = osc.iloc[i]
        bullish = (not np.isnan(o)) and (o >= 50)
        if bullish:
            val = prev if ut < prev else ut
        else:
            val = prev if dt > prev else dt
        out[i] = val
        prev = val
    return pd.Series(out, index=df.index, name='AlphaTrend')

In [ ]:
def compute_signals(df):
    """Returns df copy with AT, BuySig, SellSig columns. Faithful Pine port:
       buySignalk  = ta.crossover (AlphaTrend, AlphaTrend[2])
       sellSignalk = ta.crossunder(AlphaTrend, AlphaTrend[2])
    """
    at = alphatrend(df)
    df = df.copy()
    df['AT'] = at
    a0, a1, a2, a3 = at, at.shift(1), at.shift(2), at.shift(3)
    df['BuySig']  = ((a0 > a2)  & (a1 <= a3)).fillna(False)
    df['SellSig'] = ((a0 < a2)  & (a1 >= a3)).fillna(False)
    return df

def build_events(per_ticker):
    """Flatten per-ticker signals into a single time-sorted event list.
    Signal at bar i  →  execution at bar i+1 open (no look-ahead).
    Tie-breaking at the same timestamp: SELL before BUY (free capital first),
    then alphabetical by ticker for determinism.
    """
    events = []
    for ticker, df in per_ticker.items():
        opens = df['Open'].values
        idx   = df.index
        bs    = df['BuySig'].values
        ss    = df['SellSig'].values
        for i in range(len(df) - 1):
            if bs[i]:
                events.append((idx[i+1], 'BUY',  ticker, float(opens[i+1])))
            if ss[i]:
                events.append((idx[i+1], 'SELL', ticker, float(opens[i+1])))
    events.sort(key=lambda e: (e[0], 0 if e[1] == 'SELL' else 1, e[2]))
    return events

def portfolio_simulate(events, per_ticker,
                       start_balance=START_BAL,
                       leverage=LEVERAGE,
                       comm=COMM_RATE):
    """Single-position-at-a-time compounding simulator."""
    balance = start_balance
    in_pos  = False
    cur     = None
    trades  = []
    # balance_curve: list of (timestamp, balance) snapshots; first point = backtest start.
    first_time = events[0][0] if events else None
    balance_curve = [(first_time, balance)]

    for time, kind, ticker, price in events:
        if not in_pos and kind == 'BUY':
            if balance <= 0:
                continue
            notional = balance * leverage
            entry_comm = comm * notional
            cur = {
                'ticker': ticker,
                'entry_time': time,
                'entry_price': price,
                'notional': notional,
                'entry_comm': entry_comm,
                'balance_at_entry': balance,
            }
            in_pos = True

        elif in_pos and kind == 'SELL' and ticker == cur['ticker']:
            ret = price / cur['entry_price'] - 1.0
            exit_notional = cur['notional'] * (price / cur['entry_price'])
            exit_comm     = comm * exit_notional
            total_comm    = cur['entry_comm'] + exit_comm
            gross_pnl     = ret * cur['notional']
            net_pnl       = gross_pnl - total_comm
            balance      += net_pnl

            trades.append({
                'ticker':           cur['ticker'],
                'entry_time':       cur['entry_time'],
                'entry_price':      cur['entry_price'],
                'exit_time':        time,
                'exit_price':       price,
                'notional':         cur['notional'],
                'balance_at_entry': cur['balance_at_entry'],
                'return_pct':       ret * 100.0,
                'gross_pnl':        gross_pnl,
                'commission':       total_comm,
                'net_pnl':          net_pnl,
                'balance_after':    balance,
            })
            balance_curve.append((time, balance))
            in_pos = False
            cur    = None

    open_trade = None
    if in_pos:
        last_close = float(per_ticker[cur['ticker']]['Close'].iloc[-1])
        ret = last_close / cur['entry_price'] - 1.0
        exit_notional = cur['notional'] * (last_close / cur['entry_price'])
        exit_comm_est = comm * exit_notional
        unreal_net    = ret * cur['notional'] - cur['entry_comm'] - exit_comm_est
        open_trade = {
            **cur,
            'last_price':          last_close,
            'unrealized_pct':      ret * 100.0,
            'unrealized_net_pnl':  unreal_net,
            'projected_balance':   balance + unreal_net,
        }

    return trades, balance_curve, open_trade

In [ ]:
results = {}
for t in TICKERS:
    print(f'Fetching {t}...')
    raw = fetch_1h(t)
    if raw.empty:
        print(f'  no data, skipping')
        continue
    bars = resample_to_5h(raw)
    sig  = compute_signals(bars)
    results[t] = sig
    n_buy  = int(sig['BuySig'].sum())
    n_sell = int(sig['SellSig'].sum())
    print(f'  5H bars: {len(bars)} | raw BUY: {n_buy} | raw SELL: {n_sell}')

events = build_events(results)
print(f'\nTotal raw events across all tickers: {len(events)}')

trades, balance_curve, open_trade = portfolio_simulate(events, results)
trades_df = pd.DataFrame(trades)

print(f'\nStarting balance:  ${START_BAL:,.2f}')
print(f'Closed trades:     {len(trades_df)}')
print(f'Open position:     {open_trade["ticker"] if open_trade else "None"}')
final_bal = balance_curve[-1][1]
total_ret = (final_bal / START_BAL - 1.0) * 100.0
print(f'Final balance:     ${final_bal:,.2f}  ({total_ret:+.2f}% total return)')
if open_trade:
    print(f'Projected (incl unrealized): ${open_trade["projected_balance"]:,.2f}')

## Trade log (all tickers, all closed trades)

In [ ]:
if trades_df.empty:
    print('No closed trades.')
else:
    cols = ['ticker', 'entry_time', 'entry_price', 'exit_time', 'exit_price',
            'notional', 'balance_at_entry', 'return_pct',
            'gross_pnl', 'commission', 'net_pnl', 'balance_after']
    disp = trades_df[cols].copy()
    disp['entry_time'] = pd.to_datetime(disp['entry_time']).dt.strftime('%Y-%m-%d %H:%M')
    disp['exit_time']  = pd.to_datetime(disp['exit_time']).dt.strftime('%Y-%m-%d %H:%M')
    for c in ('entry_price', 'exit_price', 'notional', 'balance_at_entry',
              'return_pct', 'gross_pnl', 'commission', 'net_pnl', 'balance_after'):
        disp[c] = disp[c].round(2)
    display(disp)

## Per-ticker summary

In [ ]:
def summarize(group):
    n = len(group)
    wins   = group[group['net_pnl'] > 0]
    losses = group[group['net_pnl'] <= 0]
    return pd.Series({
        'trades':       n,
        'wins':         len(wins),
        'losses':       len(losses),
        'win_rate_%':   round(100.0 * len(wins) / n, 1) if n else 0.0,
        'avg_win_$':    round(wins['net_pnl'].mean(), 2)   if len(wins) else 0.0,
        'avg_loss_$':   round(losses['net_pnl'].mean(), 2) if len(losses) else 0.0,
        'best_$':       round(group['net_pnl'].max(), 2)   if n else 0.0,
        'worst_$':      round(group['net_pnl'].min(), 2)   if n else 0.0,
        'total_net_$':  round(group['net_pnl'].sum(), 2)   if n else 0.0,
        'total_comm_$': round(group['commission'].sum(), 2) if n else 0.0,
    })

if trades_df.empty:
    print('No closed trades.')
else:
    by_ticker = trades_df.groupby('ticker').apply(summarize)
    overall   = pd.DataFrame({'ALL': summarize(trades_df)}).T
    summary   = pd.concat([by_ticker, overall])
    display(summary)

    # Portfolio-level snapshot
    final_bal = balance_curve[-1][1]
    realized_ret = (final_bal / START_BAL - 1.0) * 100.0
    portfolio = pd.DataFrame([{
        'starting_balance_$':  round(START_BAL, 2),
        'final_balance_$':     round(final_bal, 2),
        'realized_return_%':   round(realized_ret, 2),
        'closed_trades':       len(trades_df),
        'leverage':            LEVERAGE,
        'open_position':       open_trade['ticker'] if open_trade else 'None',
        'projected_balance_$': round(open_trade['projected_balance'], 2) if open_trade else round(final_bal, 2),
    }])
    print('\nPortfolio-level summary:')
    display(portfolio)

## Open positions (still in trade at end of data)

In [ ]:
if open_trade:
    op = pd.DataFrame([{
        'ticker':            open_trade['ticker'],
        'entry_time':        pd.to_datetime(open_trade['entry_time']).strftime('%Y-%m-%d %H:%M'),
        'entry_price':       round(open_trade['entry_price'], 2),
        'last_price':        round(open_trade['last_price'], 2),
        'notional':          round(open_trade['notional'], 2),
        'balance_at_entry':  round(open_trade['balance_at_entry'], 2),
        'unrealized_%':      round(open_trade['unrealized_pct'], 2),
        'unrealized_net_$':  round(open_trade['unrealized_net_pnl'], 2),
        'projected_balance': round(open_trade['projected_balance'], 2),
    }])
    display(op)
else:
    print('No open positions at end of data.')

## Charts — price + AlphaTrend + signals, plus equity curve

In [ ]:
def plot_ticker(ticker, df, executed_trades):
    """Price + AlphaTrend line + only the trades actually executed by the
    portfolio simulator (signals skipped while another position was open
    are NOT shown — that matches what really happened)."""
    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
        name='Price', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=df.index, y=df['AT'], name='AlphaTrend',
        line=dict(color='#0022FC', width=2),
    ))

    if not executed_trades.empty:
        fig.add_trace(go.Scatter(
            x=executed_trades['entry_time'], y=executed_trades['entry_price'],
            mode='markers', name='BUY (executed)',
            marker=dict(symbol='triangle-up', color='#0022FC', size=14,
                        line=dict(color='white', width=1)),
        ))
        fig.add_trace(go.Scatter(
            x=executed_trades['exit_time'], y=executed_trades['exit_price'],
            mode='markers', name='SELL (executed)',
            marker=dict(symbol='triangle-down', color='#80000B', size=14,
                        line=dict(color='white', width=1)),
        ))

    fig.update_layout(
        title=f'{ticker} — 5H + AlphaTrend (executed trades only)',
        height=520, showlegend=True,
        xaxis_rangeslider_visible=False,
        margin=dict(l=40, r=20, t=50, b=30),
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])])
    return fig

for t in TICKERS:
    if t not in results:
        continue
    df = results[t]
    tr = trades_df[trades_df['ticker'] == t] if not trades_df.empty else pd.DataFrame()
    plot_ticker(t, df, tr).show()

## Portfolio balance over time

In [ ]:
if not balance_curve or len(balance_curve) <= 1:
    print('No closed trades — balance unchanged from starting value.')
else:
    times = [t for t, _ in balance_curve]
    bals  = [b for _, b in balance_curve]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=times, y=bals, mode='lines+markers',
        name='Cash balance', line=dict(color='#00874c', width=2),
        hovertemplate='%{x|%Y-%m-%d %H:%M}<br>$%{y:,.2f}<extra></extra>',
    ))
    fig.add_hline(
        y=START_BAL, line=dict(color='gray', dash='dash'),
        annotation_text=f'Start ${START_BAL:,.0f}', annotation_position='top left',
    )

    if not trades_df.empty:
        markers = trades_df[['exit_time', 'ticker', 'balance_after']].copy()
        fig.add_trace(go.Scatter(
            x=markers['exit_time'], y=markers['balance_after'],
            mode='markers', name='Trade close',
            text=markers['ticker'], hovertemplate='%{text} → $%{y:,.2f}<extra></extra>',
            marker=dict(symbol='circle', size=8, color='#0022FC',
                        line=dict(color='white', width=1)),
        ))

    final_bal = bals[-1]
    total_ret = (final_bal / START_BAL - 1.0) * 100.0
    fig.update_layout(
        title=f'Portfolio balance — start ${START_BAL:,.0f} → end ${final_bal:,.2f} ({total_ret:+.2f}%)',
        height=460, margin=dict(l=40, r=20, t=60, b=30),
        xaxis_title='Time', yaxis_title='Balance ($)',
    )
    fig.show()